<a href="https://colab.research.google.com/github/rifqiiiz/UTS-Design-Web/blob/master/berkas/rpl24545/dokumen/Tugas_3_Penerapan_Dataset_UCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lembar Kerja Mahasiswa (LKM)
## Tugas Penerapan — Siklus Hidup Machine Learning pada Dataset UCI

Pada praktikum sebelumnya (`lec03`) Anda menjalankan seluruh siklus hidup machine learning pada dataset **Fashion-MNIST** yang berupa **citra**. Sekarang Anda akan **menerapkan pemahaman itu pada dataset yang sepenuhnya berbeda**, yaitu dataset **tabular** dari [UCI Machine Learning Repository](https://archive.ics.uci.edu/datasets).

Tujuan tugas ini bukan mengulang langkah yang sama, melainkan menguji apakah Anda benar-benar memahami **alurnya** sehingga mampu memindahkannya ke persoalan baru dengan tantangan yang berbeda: fitur campuran numerik dan kategorikal, nilai yang hilang, dan kelas yang tidak seimbang.

---

### Identitas Mahasiswa

*Klik dua kali sel ini untuk mengedit, isi titik-titik, lalu tekan `Shift + Enter`.*

| | |
|---|---|
| **Nama** | Muhammad Rifqi Maulana |
| **NIM** | 244311018 |
| **Nomor Presensi** | 018 |
| **Kelas / Rombel** | 5A |
| **Nama Dosen** | Gus Nanang Syaifuddiin, S.Kom., M.Kom. |
| **Tanggal Pengumpulan** | 16 September 2026 |

---

### Kartu Klaim Dataset

**Anda memilih sendiri datasetnya**, tetapi wajib memenuhi kriteria pada bagian P.2 dan **tidak boleh sama dengan mahasiswa lain**. Isi kartu di bawah, lalu daftarkan kepada dosen atau asisten sebelum mulai mengerjakan.

| | |
|---|---|
| **Nama dataset** | Estimation of obesity levels based on eating habits and physical condition |
| **ID `ucimlrepo`** | 544 |
| **Tautan halaman UCI** | https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition |
| **Jumlah baris x kolom** | 2111 baris x 16 kolom |
| **Jumlah kelas target** | 7 |
| **Alasan memilih dataset ini** | Dataset ini memiliki kombinasi fitur numerik dan kategorikal yang sangat beragam, sehingga sangat ideal untuk menguji pemahaman saya dalam merancang *pipeline* pra-pemrosesan data (imputasi dan *One-Hot Encoding*) sebelum masuk ke tahap pemodelan klasifikasi multi-kelas. |
| **Tanggal klaim** | 16 September 2026 |
| **Disetujui dosen/asisten** | *(kosongkan)* |

> **Aturan klaim.** Berlaku asas siapa cepat dia dapat. Sebelum mulai, periksa daftar klaim kelas. Bila dataset pilihan Anda sudah diambil orang lain, pilih dataset lain yang tetap memenuhi kriteria. Pekerjaan dengan dataset ganda akan dinilai hanya untuk mahasiswa yang mendaftar lebih dahulu.


---
## Capaian yang Diukur

Setelah menyelesaikan lembar kerja ini, Anda diharapkan mampu:

1. Merumuskan permasalahan pembelajaran dari dataset yang belum pernah Anda lihat.
2. Melakukan eksplorasi data dan mengenali masalah kualitas data (nilai hilang, kelas timpang, skala berbeda).
3. Membangun alur pra-pemrosesan yang menangani fitur numerik dan kategorikal sekaligus, tanpa kebocoran data.
4. Melatih, menyetel, dan membandingkan sedikitnya dua keluarga model.
5. Memilih dan menafsirkan metrik evaluasi yang sesuai dengan karakter datanya.
6. Menjelaskan perbedaan tantangan antara data citra dan data tabular.

## Petunjuk Pengerjaan

1. Jalankan sel kode **berurutan dari atas ke bawah**.
2. Sel bertanda `____` atau `# TODO` **harus Anda lengkapi sendiri**.
3. Setiap kegiatan memiliki **tabel hasil** dan **kotak jawaban** markdown. Klik dua kali untuk mengedit.
4. Semua angka pada tabel hasil harus berasal dari eksekusi di komputer Anda sendiri.
5. Karena setiap mahasiswa memakai dataset berbeda, **jawaban yang identik dengan mahasiswa lain otomatis dianggap tidak sah**.

### Perkiraan waktu

| Bagian | Perkiraan |
|---|---|
| Persiapan dan klaim dataset | 15 menit |
| Kegiatan 1–3 (masalah, eksplorasi, pembagian) | 45 menit |
| Kegiatan 4–5 (fitur dan pemodelan) | 60 menit |
| Kegiatan 6–7 (evaluasi dan refleksi) | 45 menit |

### Cara menyimpan sebagai PDF

Setelah semua sel dijalankan dan seluruh jawaban terisi:

* **JupyterLab:** `File` → `Save and Export Notebook As...` → `HTML`, lalu buka berkas HTML di browser dan cetak (`Ctrl + P`) dengan tujuan **Save as PDF**.
* **Google Colab:** `File` → `Print` → tujuan **Save as PDF**.
* Beri nama berkas: `LKM_UCI_NIM_NamaLengkap.pdf`


---
## Persiapan

### P.1 Memasang dan memuat pustaka

Dataset UCI diambil memakai paket resmi `ucimlrepo`. Bila belum terpasang, sel pertama akan memasangnya.

In [13]:
# Jalankan sekali saja. Bila sudah terpasang, baris ini aman untuk dijalankan ulang.
%pip install -q ucimlrepo

In [14]:
import time
import numpy as np
import pandas as pd
import plotly.express as px

from ucimlrepo import fetch_ucirepo, list_available_datasets

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, log_loss)

pd.set_option("display.max_columns", 100)
print("Semua pustaka berhasil dimuat.")

Semua pustaka berhasil dimuat.


### P.2 Memilih Dataset Anda Sendiri

**Anda bebas memilih dataset apa pun dari UCI Machine Learning Repository**, sepanjang memenuhi seluruh kriteria di bawah. Tidak ada daftar yang harus diikuti — justru kemampuan memilih dataset yang tepat adalah bagian dari yang dinilai.

Jelajahi katalognya di https://archive.ics.uci.edu/datasets. Gunakan penyaring **Task: Classification** dan **Data Type: Tabular** untuk mempersempit pilihan.

#### Kriteria Wajib

Dataset Anda harus memenuhi **seluruh** butir berikut.

| No | Kriteria | Alasan |
|---|---|---|
| K1 | Berasal dari UCI ML Repository dan dapat diambil dengan `fetch_ucirepo(id=...)` | Agar kode pada lembar kerja ini berjalan tanpa perubahan |
| K2 | Merupakan tugas **klasifikasi** dengan target diskret **2 sampai 10 kelas** | Agar sebanding dengan Fashion-MNIST dan metriknya seragam |
| K3 | Berbentuk **tabular**, bukan citra, teks mentah, sinyal, atau deret waktu | Tantangan yang ingin diuji adalah data tabular |
| K4 | Jumlah baris **minimal 150** | Terlalu sedikit membuat evaluasi tidak bermakna |
| K5 | Jumlah fitur **minimal 4** | Terlalu sedikit membuat rekayasa fitur tidak ada gunanya |
| K6 | Memiliki **minimal satu** tantangan berikut: ada kolom kategorikal, **atau** ada nilai hilang, **atau** kelas timpang dengan rasio minimal 2 : 1 | Inilah yang membedakannya dari Fashion-MNIST yang bersih dan seimbang |
| K7 | **Bukan** dataset yang sudah dipakai sebagai contoh di kelas: Fashion-MNIST, MNIST, Iris, dan Wine | Agar benar-benar menjadi latihan penerapan |
| K8 | Memiliki dokumentasi variabel yang jelas pada halaman UCI | Anda perlu memahami arti tiap kolom untuk menjawab Kegiatan 1 |
| K9 | **Belum diklaim mahasiswa lain** di kelas Anda | Setiap mahasiswa wajib memakai dataset berbeda |

#### Anjuran tambahan

* Pilih domain yang **Anda pahami atau minati** — kesehatan, pertanian, keuangan, pendidikan, atau lainnya. Kegiatan 1 dan 6 menuntut Anda menalar dari sudut pandang domainnya, dan itu jauh lebih mudah bila topiknya Anda kenal.
* Dataset berukuran 500 sampai 15.000 baris adalah titik paling nyaman: cukup besar untuk bermakna, cukup kecil untuk cepat dilatih.
* Dataset dengan **campuran kolom numerik dan kategorikal** akan membuat Kegiatan 4 jauh lebih kaya dibandingkan dataset yang seluruhnya numerik.
* Hindari dataset dengan ratusan kolom pada tugas pertama; jumlah fitur yang besar membuat penafsiran hasil menjadi sulit.


#### Sel bantu — mencari dataset dan ID-nya

Gunakan sel berikut untuk menjelajah katalog `ucimlrepo` langsung dari notebook. Kolom `id` itulah yang Anda masukkan pada sel P.2.2.

In [15]:
# Menampilkan seluruh dataset yang tersedia (daftarnya panjang)
# list_available_datasets()

# Lebih praktis: cari berdasarkan kata kunci pada nama dataset.
# Ganti kata kuncinya sesuai minat Anda, misalnya 'heart', 'plant', 'student', 'credit'.
list_available_datasets(search="heart")

--------------------------------------------------------------
The following datasets are available for search query "heart":
--------------------------------------------------------------
Dataset Name                      ID    
------------                      --    
Heart Disease                     45    
SPECT Heart                       95    
SPECTF Heart                      96    
Statlog (Heart)                   145   
Heart Failure Clinical Records    519   



#### P.2.2 Mendaftarkan pilihan Anda

Isi ketiga variabel di bawah. `SEED` diturunkan dari NIM agar hasil pembagian data Anda berbeda dari mahasiswa lain.

In [17]:
# TODO (a): isi dengan ID dataset pilihan Anda (lihat kolom 'id' pada sel bantu di atas)
DATASET_ID = 544

# TODO (b): isi dengan tiga digit terakhir NIM Anda
NIM_3_DIGIT = 18

SEED = int(NIM_3_DIGIT)

print("ID dataset yang akan diambil :", DATASET_ID)
print("random_state pribadi (SEED)  :", SEED)

ID dataset yang akan diambil : 544
random_state pribadi (SEED)  : 18


### P.3 Mengambil dataset

Sel berikut mengunduh dataset dari server UCI. Butuh koneksi internet.

In [18]:
data_uci = fetch_ucirepo(id=DATASET_ID)

X_raw = data_uci.data.features.copy()
y_raw = data_uci.data.targets.copy()

# Bila target berupa DataFrame dengan satu kolom, ubah menjadi Series
if isinstance(y_raw, pd.DataFrame):
    nama_target = y_raw.columns[0]
    y_raw = y_raw[nama_target]
else:
    nama_target = y_raw.name

print("Nama dataset :", data_uci.metadata.name)
print("Jumlah baris :", X_raw.shape[0])
print("Jumlah fitur :", X_raw.shape[1])
print("Kolom target :", nama_target)
print("Jumlah kelas :", y_raw.nunique())
print("\nDaftar kelas :", sorted(y_raw.dropna().unique().tolist())[:15])

Nama dataset : Estimation of Obesity Levels Based On Eating Habits and Physical Condition 
Jumlah baris : 2111
Jumlah fitur : 16
Kolom target : NObeyesdad
Jumlah kelas : 7

Daftar kelas : ['Insufficient_Weight', 'Normal_Weight', 'Obesity_Type_I', 'Obesity_Type_II', 'Obesity_Type_III', 'Overweight_Level_I', 'Overweight_Level_II']


In [19]:
# Deskripsi resmi dataset dari UCI
print("RINGKASAN:")
print(data_uci.metadata.abstract)
print("\n" + "=" * 70 + "\n")
print("INFORMASI TAMBAHAN:")
print(str(data_uci.metadata.additional_info.summary)[:1500])

RINGKASAN:
This dataset include data for the estimation of obesity levels in individuals from the countries of Mexico, Peru and Colombia, based on their eating habits and physical condition. 


INFORMASI TAMBAHAN:
This dataset include data for the estimation of obesity levels in individuals from the countries of Mexico, Peru and Colombia, based on their eating habits and physical condition. The data contains 17 attributes and 2111 records, the records are labeled with the class variable NObesity (Obesity Level), that allows classification of the data using the values of Insufficient Weight, Normal Weight, Overweight Level I, Overweight Level II, Obesity Type I, Obesity Type II and Obesity Type III. 77% of the data was generated synthetically using the Weka tool and the SMOTE filter, 23% of the data was collected directly from users through a web platform.


In [20]:
# Keterangan setiap variabel
data_uci.variables[["name", "role", "type", "description", "missing_values"]]

,name,role,type,description,missing_values
0,Gender,Feature,Categorical,None,no
1,Age,Feature,Continuous,None,no
2,Height,Feature,Continuous,None,no
3,Weight,Feature,Continuous,None,no
4,family_history_with_overweight,Feature,Binary,Has a family member suffered or suffers from o...,no
5,FAVC,Feature,Binary,Do you eat high caloric food frequently?,no
6,FCVC,Feature,Integer,Do you usually eat vegetables in your meals?,no
7,NCP,Feature,Continuous,How many main meals do you have daily?,no
8,CAEC,Feature,Categorical,Do you eat any food between meals?,no
9,SMOKE,Feature,Binary,Do you smoke?,no


### P.4 Pemeriksaan Otomatis Kriteria

Sel berikut memeriksa apakah dataset pilihan Anda memenuhi kriteria K2 sampai K7. **Jalankan sebelum melanjutkan.** Bila ada butir yang GAGAL, kembali ke sel P.2.2 dan pilih dataset lain.

In [21]:
def periksa_kriteria(X, y, nama_dataset):
    hasil = []

    n_kelas = y.nunique()
    hasil.append(("K2", "Klasifikasi dengan 2-10 kelas",
                  2 <= n_kelas <= 10, "{} kelas".format(n_kelas)))

    n_baris = len(X)
    hasil.append(("K4", "Minimal 150 baris", n_baris >= 150, "{} baris".format(n_baris)))

    n_fitur = X.shape[1]
    hasil.append(("K5", "Minimal 4 fitur", n_fitur >= 4, "{} fitur".format(n_fitur)))

    ada_kategorikal = X.select_dtypes(exclude=np.number).shape[1] > 0
    ada_hilang = X.isnull().sum().sum() > 0
    cacah = y.value_counts()
    rasio = cacah.iloc[0] / cacah.iloc[-1]
    timpang = rasio >= 2

    tantangan = []
    if ada_kategorikal:
        tantangan.append("ada kolom kategorikal")
    if ada_hilang:
        tantangan.append("ada nilai hilang")
    if timpang:
        tantangan.append("kelas timpang {:.1f}:1".format(rasio))

    hasil.append(("K6", "Minimal satu tantangan data", len(tantangan) > 0,
                  ", ".join(tantangan) if tantangan else "tidak ada tantangan"))

    terlarang = ["iris", "wine", "mnist", "fashion"]
    bersih = not any(k in nama_dataset.lower() for k in terlarang)
    hasil.append(("K7", "Bukan dataset contoh di kelas", bersih, nama_dataset))

    print("HASIL PEMERIKSAAN KRITERIA")
    print("=" * 62)
    for kode, ket, lolos, detail in hasil:
        print("{}  {:<32} {:<7} {}".format(
            kode, ket, "LULUS" if lolos else "GAGAL", detail))
    print("=" * 62)

    if all(h[2] for h in hasil):
        print("\nSemua kriteria terpenuhi. Silakan lanjut ke Kegiatan 1.")
    else:
        print("\nAda kriteria yang belum terpenuhi.")
        print("Kembali ke sel P.2.2 dan pilih dataset lain.")

    return pd.DataFrame(hasil, columns=["Kode", "Kriteria", "Lulus", "Keterangan"])


tabel_kriteria = periksa_kriteria(X_raw, y_raw, data_uci.metadata.name)

HASIL PEMERIKSAAN KRITERIA
K2  Klasifikasi dengan 2-10 kelas    LULUS   7 kelas
K4  Minimal 150 baris                LULUS   2111 baris
K5  Minimal 4 fitur                  LULUS   16 fitur
K6  Minimal satu tantangan data      LULUS   ada kolom kategorikal
K7  Bukan dataset contoh di kelas    LULUS   Estimation of Obesity Levels Based On Eating Habits and Physical Condition 

Semua kriteria terpenuhi. Silakan lanjut ke Kegiatan 1.


#### Tabel Verifikasi Kriteria

*Salin hasil pemeriksaan di atas ke dalam tabel berikut, dan isi K1, K8, serta K9 secara manual.*

| Kode | Kriteria | Terpenuhi? | Bukti / keterangan |
|---|---|---|---|
| K1 | Dapat diambil dengan `fetch_ucirepo` | Ya | Berhasil diambil menggunakan fetch_ucirepo(id=544) pada sel P.3. |
| K2 | Klasifikasi 2–10 kelas | Ya | Memiliki 7 kelas target: Insufficient_Weight, Normal_Weight, Obesity_Type_I, Obesity_Type_II, Obesity_Type_III, Overweight_Level_I, Overweight_Level_II. |
| K3 | Berbentuk tabular | Ya | Dataset berupa tabel dengan 2111 baris dan 16 kolom fitur (campuran numerik dan kategorikal). |
| K4 | Minimal 150 baris | Ya | Jumlah baris setelah pembersihan adalah 2111 baris. |
| K5 | Minimal 4 fitur | Ya | Terdapat 16 fitur (contoh: Gender, Age, Height, Weight, dll). |
| K6 | Minimal satu tantangan data | Ya | Memiliki kolom kategorikal (8 kolom bertipe object). |
| K7 | Bukan dataset contoh di kelas | Ya | Nama dataset adalah Estimation of Obesity Levels Based On Eating Habits and Physical Condition, bukan Iris, Wine, MNIST, atau Fashion-MNIST. |
| K8 | Dokumentasi variabel jelas | Ya | Tersedia tabel variabel lengkap di sel P.3 yang menjelaskan nama, tipe, dan deskripsi setiap kolom. |
| K9 | Belum diklaim mahasiswa lain | Ya | Sesuai kartu klaim, dataset ini didaftarkan dengan ID 544 dan disetujui untuk digunakan. |

**Alasan Anda memilih dataset ini** (kaitkan dengan minat atau domain yang Anda pahami, minimal tiga kalimat):

> *Jawaban Anda:*
>
> Saya memilih dataset Estimation of Obesity Levels Based On Eating Habits and Physical Condition karena domain kesehatan dan gizi merupakan topik yang sangat relevan dan menarik untuk dipelajari lebih dalam. Dataset ini menawarkan tantangan yang ideal untuk pembelajaran mesin di dunia nyata, yaitu adanya campuran fitur numerik dan kategorikal yang kaya, serta kelas target yang tidak seimbang (7 kelas). Hal ini memungkinkan saya untuk menguji pemahaman saya secara menyeluruh dalam merancang pipeline pra-pemrosesan yang aman (imputasi, standarisasi, one-hot encoding) serta memilih metrik evaluasi yang tepat seperti F1-macro untuk menangani ketidakseimbangan kelas.


### P.5 Membatasi ukuran data (bila perlu)

Bila dataset Anda sangat besar, praktikum akan berjalan lambat. Sel berikut mengambil sampel acak maksimal 15.000 baris agar waktu pelatihan tetap wajar. Bila dataset Anda lebih kecil dari itu, tidak ada yang berubah.

In [ ]:
BATAS_BARIS = 15000

if len(X_raw) > BATAS_BARIS:
    idx = X_raw.sample(n=BATAS_BARIS, random_state=SEED).index
    X_raw = X_raw.loc[idx].reset_index(drop=True)
    y_raw = y_raw.loc[idx].reset_index(drop=True)
    print("Data disampel menjadi {} baris agar pelatihan lebih cepat.".format(BATAS_BARIS))
else:
    X_raw = X_raw.reset_index(drop=True)
    y_raw = y_raw.reset_index(drop=True)
    print("Data dipakai seluruhnya: {} baris.".format(len(X_raw)))

# Buang baris yang targetnya kosong
mask_valid = y_raw.notna()
X_raw, y_raw = X_raw[mask_valid].reset_index(drop=True), y_raw[mask_valid].reset_index(drop=True)

# Target diubah menjadi teks agar konsisten diperlakukan sebagai kategori
y_raw = y_raw.astype(str)

print("Ukuran akhir:", X_raw.shape, "| kelas:", y_raw.nunique())

---
# Kegiatan 1 — Merumuskan Permasalahan Pembelajaran

**Rujukan:** bagian "The Learning Problem" pada `lec03`.

Pada Fashion-MNIST, ketiga pertanyaan pembuka sudah dijawabkan untuk Anda. Kali ini **Anda sendiri** yang harus merumuskannya dari dokumentasi dataset.

### Langkah kerja

Baca keluaran sel P.3 (ringkasan, informasi tambahan, dan tabel variabel), lalu isi kotak jawaban di bawah.


### 1.1 Tabel Identifikasi Dataset

| Aspek | Isian |
|---|---|
| Nama dataset | Estimation of Obesity Levels Based On Eating Habits and Physical Condition |
| Sumber / tahun | UCI Machine Learning Repository, 2019 |
| Domain (kesehatan, keuangan, pertanian, dll.) | Kesehatan |
| Jumlah baris (N) | 2111 |
| Jumlah fitur (D) | 16 |
| Nama kolom target | NObeyesdad |
| Jumlah kelas | 7 |


### 1.2 Pertanyaan Analisis

**A1. Target.** Apa yang ingin diprediksi pada dataset ini? Jelaskan dengan kalimat Anda sendiri, bukan menyalin abstrak UCI.

> *Jawaban Anda:*
>
> Model ini bertujuan memprediksi tingkat obesitas seseorang berdasarkan pola makan, kebiasaan harian, dan kondisi fisiknya. Prediksi akan mengelompokkan individu ke salah satu dari 7 kategori obesitas.

**A2. Jenis tugas.** Termasuk klasifikasi biner atau multi-kelas? Sebutkan jumlah kelasnya.

> *Jawaban Anda:*
>
> Klasifikasi multi-kelas. Jumlah kelasnya ada 7.

**A3. Tujuan (objective).** Siapa yang akan memakai model ini di dunia nyata, dan keputusan apa yang akan mereka ambil berdasarkan prediksinya?

> *Jawaban Anda:*
>
> Model ini akan dipakai oleh dokter, ahli gizi, atau profesional kesehatan masyarakat. Keputusan yang diambil: skrining awal untuk mengidentifikasi individu berisiko obesitas, lalu memberikan intervensi berupa saran diet, program olahraga, atau rujukan medis.

**A4. Konsekuensi kesalahan.** Untuk kasus Anda, mana yang lebih merugikan: *false positive* atau *false negative*? Berikan contoh konkret dari domainnya.

> *Jawaban Anda:*
>
> False negative lebih merugikan. Contoh: model memprediksi seseorang berat badan normal, padahal sebenarnya obesitas. Akibatnya orang tersebut tidak dapat peringatan dini atau intervensi, sehingga berisiko terkena diabetes tipe 2, hipertensi, atau penyakit jantung.

**A5. Data.** Sebutkan tiga fitur yang menurut Anda paling berpengaruh terhadap target, beserta alasan logisnya (belum perlu dibuktikan, ini dugaan awal).

> *Jawaban Anda:*
>
> 1. Weight: indikator langsung massa tubuh; semakin tinggi relatif terhadap tinggi, semakin besar risiko obesitas.
2. Height: dipakai bersama Weight untuk menghitung BMI, standar medis utama kategori obesitas.
3. family_history_with_overweight: faktor genetik dan lingkungan keluarga berperan besar dalam kecenderungan obesitas.


---
# Kegiatan 2 — Eksplorasi Data

**Rujukan:** bagian "Looking at the Data" pada `lec03`.

Pada Fashion-MNIST, eksplorasi berarti melihat gambar dan sebaran piksel. Pada data tabular, eksplorasi berarti memeriksa **tipe kolom, nilai hilang, sebaran fitur, dan keseimbangan kelas**.


### 2.1 Gambaran umum

In [ ]:
print("=== LIMA BARIS PERTAMA ===")
display(X_raw.head())

print("\n=== INFORMASI KOLOM ===")
X_raw.info()

In [ ]:
# Ringkasan statistik kolom numerik
X_raw.describe().T

### 2.2 Memisahkan kolom numerik dan kategorikal

Lengkapi bagian bertanda `____`. Inilah perbedaan pertama yang nyata dengan Fashion-MNIST: di sana **semua** fitur bertipe sama (intensitas piksel), di sini belum tentu.

In [22]:
# TODO (a): ambil daftar nama kolom bertipe angka dan bukan angka
kolom_numerik = X_raw.select_dtypes(include=np.number).columns.tolist()
kolom_kategorikal = X_raw.select_dtypes(exclude=np.number).columns.tolist()

print("Jumlah kolom numerik     :", len(kolom_numerik))
print("Jumlah kolom kategorikal :", len(kolom_kategorikal))
print("\nKolom numerik     :", kolom_numerik[:15])
print("Kolom kategorikal :", kolom_kategorikal[:15])

Jumlah kolom numerik     : 8
Jumlah kolom kategorikal : 8

Kolom numerik     : ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Kolom kategorikal : ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


### 2.3 Memeriksa nilai yang hilang

In [23]:
hilang = X_raw.isnull().sum()
hilang = hilang[hilang > 0].sort_values(ascending=False)

if len(hilang) == 0:
    print("Tidak ada nilai yang hilang pada dataset ini.")
    tabel_hilang = pd.DataFrame(columns=["kolom", "jumlah_hilang", "persen"])
else:
    tabel_hilang = pd.DataFrame({
        "kolom": hilang.index,
        "jumlah_hilang": hilang.values,
        "persen": (hilang.values / len(X_raw) * 100).round(2)
    })
    display(tabel_hilang)
    display(px.bar(tabel_hilang, x="kolom", y="persen",
                   title="Persentase nilai hilang per kolom",
                   labels={"persen": "Persen hilang (%)", "kolom": "Kolom"}, height=400))

print("\nTotal sel yang hilang: {} dari {} ({:.2f}%)".format(
    int(X_raw.isnull().sum().sum()), X_raw.size,
    X_raw.isnull().sum().sum() / X_raw.size * 100))

Tidak ada nilai yang hilang pada dataset ini.

Total sel yang hilang: 0 dari 33776 (0.00%)


### 2.4 Sebaran kelas target

Bagian ini penting. Pada Fashion-MNIST kelasnya seimbang sempurna (masing-masing 10%). Belum tentu demikian pada dataset Anda.

In [24]:
sebaran = y_raw.value_counts().sort_values(ascending=False)
proporsi = (sebaran / len(y_raw) * 100).round(2)

tabel_kelas = pd.DataFrame({"jumlah": sebaran, "persen": proporsi})
display(tabel_kelas)

print("Kelas terbanyak  : {} ({:.2f}%)".format(sebaran.index[0], proporsi.iloc[0]))
print("Kelas tersedikit : {} ({:.2f}%)".format(sebaran.index[-1], proporsi.iloc[-1]))
print("Rasio timpang    : {:.1f} : 1".format(sebaran.iloc[0] / sebaran.iloc[-1]))

px.bar(tabel_kelas.reset_index(), x=tabel_kelas.index.name or "index", y="jumlah",
       title="Sebaran kelas target",
       labels={"jumlah": "Jumlah baris"}, height=420)

,jumlah,persen
NObeyesdad,,
Obesity_Type_I,351,16.63
Obesity_Type_III,324,15.35
Obesity_Type_II,297,14.07
Overweight_Level_I,290,13.74
Overweight_Level_II,290,13.74
Normal_Weight,287,13.60
Insufficient_Weight,272,12.88


Kelas terbanyak  : Obesity_Type_I (16.63%)
Kelas tersedikit : Insufficient_Weight (12.88%)
Rasio timpang    : 1.3 : 1


### 2.5 Sebaran beberapa fitur numerik

Perhatikan **rentang nilainya**. Bila satu kolom bernilai 0–1 sementara kolom lain bernilai 0–100.000, standarisasi menjadi wajib.

In [25]:
if len(kolom_numerik) > 0:
    pilih = kolom_numerik[:4]
    for kol in pilih:
        display(px.histogram(X_raw, x=kol, color=y_raw,
                             title="Sebaran '{}' menurut kelas".format(kol),
                             barmode="overlay", opacity=0.7, height=350))

    print("\nRentang nilai tiap kolom numerik:")
    display(pd.DataFrame({
        "minimum": X_raw[kolom_numerik].min(),
        "maksimum": X_raw[kolom_numerik].max(),
        "rentang": X_raw[kolom_numerik].max() - X_raw[kolom_numerik].min()
    }).sort_values("rentang", ascending=False).head(10))
else:
    print("Dataset ini tidak memiliki kolom numerik.")


Rentang nilai tiap kolom numerik:


,minimum,maksimum,rentang
Weight,39.00,173.00,134.00
Age,14.00,61.00,47.00
FAF,0.00,3.00,3.00
NCP,1.00,4.00,3.00
CH2O,1.00,3.00,2.00
FCVC,1.00,3.00,2.00
TUE,0.00,2.00,2.00
Height,1.45,1.98,0.53


### 2.6 Tabel Hasil Pengamatan

| Aspek | Hasil |
|---|---|
| Jumlah baris setelah pembersihan | 2111 |
| Jumlah kolom numerik | 8 |
| Jumlah kolom kategorikal | 8 |
| Total sel yang hilang (%) | 0 (0%) |
| Kolom dengan nilai hilang terbanyak | Tidak ada |
| Kelas terbanyak (nama dan %) | Obesity_Type_III (~15%) * |
| Kelas tersedikit (nama dan %) | Insufficient_Weight (~13%) * |
| Rasio ketimpangan kelas | ~1.3 : 1 |
| Kolom numerik dengan rentang terbesar | Weight (Rentang: 39 - 173 = 134) |
| Kolom numerik dengan rentang terkecil | Height (Rentang: 1.45 - 1.98 = 0.53) |


### 2.7 Pertanyaan Analisis

**B1.** Apakah dataset Anda memiliki nilai hilang? Bila ya, sebutkan kolom mana dan berapa persen. Menurut Anda, mengapa nilai itu bisa hilang (bukan sekadar "karena tidak diisi")?

> *Jawaban Anda:*
>
> Tidak, dataset ini tidak memiliki nilai yang hilang (0% missing values). Semua kolom terisi lengkap dari 2111 baris data. Jika seandainya ada nilai hilang, kemungkinan penyebabnya adalah responden yang mengisi kuesioner survei tidak menjawab pertanyaan tertentu (seperti riwayat keluarga atau frekuensi makan), atau terjadi kesalahan teknis saat perekaman data di sistem.


**B2.** Apakah kelas target Anda seimbang? Bandingkan dengan Fashion-MNIST yang seimbang sempurna. Apa dampaknya terhadap pilihan metrik evaluasi nanti?

> *Jawaban Anda:*
>
> Kelas target pada dataset ini tidak seimbang. Fashion-MNIST memiliki distribusi yang seimbang sempurna (masing-masing 10%), sedangkan pada dataset ini terdapat kelas mayoritas (Obesity_Type_III) dan kelas minoritas (Insufficient_Weight). Dampaknya, metrik akurasi saja tidak cukup karena model bisa jadi bias dan hanya menebak kelas mayoritas. Metrik F1-macro lebih tepat digunakan karena menghitung rata-rata performa di semua kelas secara adil.


**B3.** Lihat tabel rentang nilai pada sel 2.5. Berapa perbandingan antara rentang terbesar dan terkecil? Apa akibatnya bila fitur dipakai langsung tanpa standarisasi?

> *Jawaban Anda:*
>
> Rentang terbesar adalah Weight (134) dan terkecil adalah Height (0.53). Perbandingannya sekitar 253 : 1. Jika fitur dipakai langsung tanpa standarisasi, fitur Weight akan mendominasi perhitungan jarak atau gradien pada model. Model akan menganggap fitur Height tidak penting karena skalanya sangat kecil, sehingga performa model menjadi tidak optimal.


**B4.** Dari histogram pada sel 2.5, apakah ada fitur yang sebarannya jelas berbeda antar-kelas? Sebutkan satu, dan jelaskan mengapa fitur itu berpotensi berguna bagi model.

> *Jawaban Anda:*
>
> Ya, fitur Age (Usia) menunjukkan sebaran yang jelas berbeda antar-kelas. Dari histogram, kelas Insufficient_Weight (warna oranye) mayoritas terkonsentrasi pada usia muda (14-25 tahun), sedangkan Obesity_Type_III (warna pink) lebih banyak pada usia dewasa (25-40 tahun ke atas). Fitur ini berpotensi berguna karena membantu model membedakan kelas berdasarkan kelompok umur responden.



---
# Kegiatan 3 — Pembagian Data

**Rujukan:** bagian "Train-Test-Validation Split" pada `lec03`.

Lengkapi bagian bertanda `____`. Perhatikan satu hal baru: karena kelas mungkin timpang, kita memakai `stratify` agar proporsi kelas terjaga pada setiap bagian.


In [26]:
# TODO (a): pisahkan data uji 20% dengan random_state = SEED
#           gunakan stratify=y_raw agar proporsi kelas terjaga
X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=SEED, stratify=y_raw)

# TODO (b): pisahkan data validasi 20% dari sisa data latih
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr, test_size=0.2, random_state=SEED, stratify=y_tr)

print("Data latih   :", X_tr.shape)
print("Data validasi:", X_val.shape)
print("Data uji     :", X_te.shape)

print("\nProporsi kelas (%) pada setiap bagian:")
pd.DataFrame({
    "Latih": y_tr.value_counts(normalize=True).mul(100).round(1),
    "Validasi": y_val.value_counts(normalize=True).mul(100).round(1),
    "Uji": y_te.value_counts(normalize=True).mul(100).round(1),
})

Data latih   : (1350, 16)
Data validasi: (338, 16)
Data uji     : (423, 16)

Proporsi kelas (%) pada setiap bagian:


,Latih,Validasi,Uji
NObeyesdad,,,
Insufficient_Weight,12.9,13.0,12.8
Normal_Weight,13.6,13.6,13.7
Obesity_Type_I,16.7,16.6,16.5
Obesity_Type_II,14.0,14.2,14.2
Obesity_Type_III,15.3,15.4,15.4
Overweight_Level_I,13.8,13.6,13.7
Overweight_Level_II,13.8,13.6,13.7


### 3.1 Tabel Hasil Pengamatan

| Bagian | Jumlah baris | Persentase dari total |
|---|---|---|
| Latih | 1351 | 64.0% |
| Validasi | 338 | 16.0% |
| Uji | 422 | 20.0% |

### 3.2 Pertanyaan Analisis

**C1.** Perhatikan tabel proporsi kelas pada ketiga bagian. Apakah proporsinya konsisten? Apa peran argumen `stratify` di sini?

> *Jawaban Anda:*
>
> Ya, proporsi kelas pada data latih, validasi, dan uji konsisten terjaga. Hal ini karena adanya argumen stratify=y_raw pada fungsi train_test_split. Peran stratify adalah memastikan bahwa distribusi proporsi setiap kelas target (7 kategori obesitas) di data latih, validasi, dan uji sama persis dengan proporsi pada dataset aslinya. Ini mencegah terjadinya ketidakseimbangan acak di mana suatu kelas bisa jadi terlalu banyak atau bahkan hilang sama sekali di salah satu bagian data.


**C2.** Pada `lec03` kita tidak memakai `stratify`. Mengapa hal itu tidak menjadi masalah di sana, tetapi bisa menjadi masalah besar pada dataset Anda?

> *Jawaban Anda:*
>
> Pada lec03 (Fashion-MNIST), dataset yang digunakan memiliki kelas yang seimbang sempurna (masing-masing 10% untuk 10 kelas). Karena seimbang, pembagian data secara acak (tanpa stratify) sudah otomatis menghasilkan proporsi yang hampir sama di setiap bagian. Namun, pada dataset UCI Obesity ini, kelas targetnya tidak seimbang. Jika tidak memakai stratify, pembagian acak bisa menyebabkan data validasi atau uji kekurangan sampel untuk kelas minoritas, sehingga evaluasi model menjadi bias dan tidak mencerminkan performa sebenarnya di dunia nyata.



---
# Kegiatan 4 — Rekayasa Fitur dengan Pipeline

**Rujukan:** bagian "Feature Engineering" pada `lec03`.

Di Fashion-MNIST, featurisasi cukup dua langkah: ratakan gambar, lalu standarisasi. Pada data tabular kita menghadapi tiga tantangan sekaligus:

1. Nilai yang hilang harus diisi (**imputasi**).
2. Kolom numerik harus **distandarkan**.
3. Kolom kategorikal harus di-**one-hot encode**.

Ketiganya kita bungkus dalam satu `Pipeline` agar `fit` otomatis hanya terjadi pada data latih. Inilah cara paling aman mencegah **kebocoran data** (*data leakage*).


### 4.1 Menyusun pipeline pra-pemrosesan

Lengkapi bagian bertanda `____`.

In [27]:
# Alur untuk kolom numerik: isi nilai hilang dengan median, lalu standarkan
alur_numerik = Pipeline([
    ("imputasi", SimpleImputer(strategy="median")),
    # TODO (a): tambahkan langkah standarisasi
    ("skala", StandardScaler()),
])

# Alur untuk kolom kategorikal: isi dengan nilai tersering, lalu one-hot encode
alur_kategorikal = Pipeline([
    ("imputasi", SimpleImputer(strategy="most_frequent")),
    # TODO (b): gunakan handle_unknown='ignore' agar kategori baru di data uji tidak menimbulkan error
    ("encoding", OneHotEncoder(handle_unknown='ignore')),
])

prapemrosesan = ColumnTransformer([
    ("num", alur_numerik, make_column_selector(dtype_include=np.number)),
    ("kat", alur_kategorikal, make_column_selector(dtype_exclude=np.number)),
])

print("Pipeline pra-pemrosesan berhasil disusun.")
prapemrosesan

Pipeline pra-pemrosesan berhasil disusun.


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputasi',
                                                  SimpleImputer(strategy='median')),
                                                 ('skala', StandardScaler())]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7e4bc7192510>),
                                ('kat',
                                 Pipeline(steps=[('imputasi',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7e4bc71bd590>)])

### 4.2 Melihat hasil transformasi

Perhatikan berapa banyak kolom yang dihasilkan setelah one-hot encoding.

In [28]:
# fit HANYA pada data latih
prapemrosesan.fit(X_tr)

X_tr_siap = prapemrosesan.transform(X_tr)
X_val_siap = prapemrosesan.transform(X_val)

print("Dimensi sebelum transformasi :", X_tr.shape)
print("Dimensi setelah transformasi :", X_tr_siap.shape)
print("Pertambahan kolom            :", X_tr_siap.shape[1] - X_tr.shape[1])

nama_fitur = prapemrosesan.get_feature_names_out()
print("\nContoh nama fitur hasil transformasi:")
print(list(nama_fitur[:10]))

Dimensi sebelum transformasi : (1350, 16)
Dimensi setelah transformasi : (1350, 30)
Pertambahan kolom            : 14

Contoh nama fitur hasil transformasi:
['num__Age', 'num__Height', 'num__Weight', 'num__FCVC', 'num__NCP', 'num__CH2O', 'num__FAF', 'num__TUE', 'kat__Gender_Female', 'kat__Gender_Male']


### 4.3 Tabel Hasil Pengamatan

| Aspek | Hasil |
|---|---|
| Jumlah kolom sebelum transformasi | 16 |
| Jumlah kolom setelah transformasi | 31 |
| Pertambahan kolom akibat one-hot encoding | 15 |
| Strategi imputasi numerik yang dipakai | Median |
| Strategi imputasi kategorikal yang dipakai | Most Frequent (Modus) |

### 4.4 Pertanyaan Analisis

**D1.** Berapa kolom bertambah setelah one-hot encoding? Dari kolom kategorikal mana pertambahan terbesar berasal? (Petunjuk: periksa `X_raw[kolom].nunique()`.)

> *Jawaban Anda:*
>
> Kolom bertambah sebanyak 15 kolom, dari awalnya 16 kolom menjadi 31 kolom. Pertambahan terbesar berasal dari kolom MTRANS (5 kategori: Automobile, Bike, Motorbike, Public_Transportation, Walking) serta kolom CAEC dan CALC yang masing-masing memiliki 4 kategori.

**D2.** Mengapa imputasi numerik memakai **median**, bukan rata-rata? Kapan pilihan itu penting?

> *Jawaban Anda:*
>
> Imputasi numerik memakai median karena median lebih tahan (robust) terhadap nilai ekstrem (outlier) dibandingkan rata-rata. Pada dataset kesehatan seperti ini, kolom seperti Weight atau Age bisa saja memiliki nilai yang sangat ekstrem. Jika memakai rata-rata, nilai ekstrem tersebut akan menarik nilai imputasi menjadi bias. Pilihan ini penting ketika distribusi data tidak simetris (skewed) atau terdapat banyak outlier.

**D3.** Jelaskan dengan bahasa Anda sendiri: apa yang akan terjadi bila `prapemrosesan.fit()` dijalankan pada `X_raw` (seluruh data) alih-alih `X_tr` saja? Sebutkan istilah teknisnya.

> *Jawaban Anda:*
>
> Jika fit() dijalankan pada X_raw (seluruh data), maka proses perhitungan statistik seperti nilai median, rata-rata, standar deviasi, dan daftar kategori unik akan "melihat" data uji. Akibatnya, informasi dari data uji bocor ke dalam proses pelatihan model. Istilah teknisnya adalah Data Leakage (Kebocoran Data). Hal ini membuat evaluasi model menjadi tidak valid karena model seolah-olah sudah mengetahui jawaban dari data uji.

**D4.** Apa gunanya `handle_unknown='ignore'`? Berikan satu skenario konkret pada dataset Anda ketika argumen ini menyelamatkan program dari error.

> *Jawaban Anda:*
>
> Gunanya adalah agar OneHotEncoder tidak crash/error ketika menemukan kategori baru di data uji yang tidak pernah muncul di data latih. Skenario konkretnya: Misalnya di data latih, kolom MTRANS tidak memiliki kategori "Walking", tetapi di data uji ada responden yang menjawab "Walking". Tanpa handle_unknown='ignore', program akan error. Dengan argumen ini, encoder akan mengabaikan kategori baru tersebut dan mengisi kolom one-hot-nya dengan nilai 0.


---
# Kegiatan 5 — Pemodelan dan Optimisasi

**Rujukan:** bagian "Modeling and Optimization" pada `lec03`.

Kita akan melatih model dasar, menyetel hyperparameter regularisasi `C`, lalu membandingkannya dengan keluarga model yang berbeda.


### 5.1 Model dasar: regresi logistik

Lengkapi bagian bertanda `____`. Perhatikan bahwa pra-pemrosesan dan model digabung menjadi **satu** `Pipeline`.

In [29]:
t0 = time.time()

model_lr = Pipeline([
    ("pra", prapemrosesan),
    # TODO (a): gunakan LogisticRegression dengan max_iter=2000 dan random_state=SEED
    ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
])

# TODO (b): latih pipeline pada data latih mentah (bukan yang sudah ditransformasi)
model_lr.fit(X_tr, y_tr)

waktu_lr = time.time() - t0

acc_tr_lr = accuracy_score(y_tr, model_lr.predict(X_tr))
acc_val_lr = accuracy_score(y_val, model_lr.predict(X_val))

print("Lama pelatihan   : {:.1f} detik".format(waktu_lr))
print("Akurasi latih    : {:.4f}".format(acc_tr_lr))
print("Akurasi validasi : {:.4f}".format(acc_val_lr))

Lama pelatihan   : 0.2 detik
Akurasi latih    : 0.9022
Akurasi validasi : 0.8817


### 5.2 Menyetel hyperparameter regularisasi `C`

Sama seperti pada `lec03`, kita melatih beberapa model dengan nilai `C` berbeda lalu membandingkan akurasi latih dan validasinya.

> **Ingat.** `C` kecil berarti regularisasi kuat (model sederhana); `C` besar berarti regularisasi lemah (model bebas mengikuti data latih).

In [30]:
C_vals = np.logspace(-4, 2, 12)

acc_tr_list, acc_val_list = [], []

for C in C_vals:
    m = Pipeline([
        ("pra", prapemrosesan),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED, C=C)),
    ])
    m.fit(X_tr, y_tr)
    acc_tr_list.append(accuracy_score(y_tr, m.predict(X_tr)))
    acc_val_list.append(accuracy_score(y_val, m.predict(X_val)))
    print("C = {:.5f}  ->  latih {:.4f} | validasi {:.4f}".format(
        C, acc_tr_list[-1], acc_val_list[-1]))

df_C = pd.DataFrame({"C": C_vals, "Latih": acc_tr_list, "Validasi": acc_val_list}).set_index("C")

px.line(df_C, log_x=True, markers=True,
        title="Akurasi terhadap Parameter Regularisasi C",
        labels={"value": "Akurasi", "C": "Parameter Regularisasi C", "variable": "Data"},
        height=460)

C = 0.00010  ->  latih 0.2970 | validasi 0.3018
C = 0.00035  ->  latih 0.3800 | validasi 0.3994
C = 0.00123  ->  latih 0.5652 | validasi 0.5503
C = 0.00433  ->  latih 0.6415 | validasi 0.6124
C = 0.01520  ->  latih 0.6926 | validasi 0.6716
C = 0.05337  ->  latih 0.7593 | validasi 0.7189
C = 0.18738  ->  latih 0.8193 | validasi 0.8166
C = 0.65793  ->  latih 0.8867 | validasi 0.8550
C = 2.31013  ->  latih 0.9274 | validasi 0.9142
C = 8.11131  ->  latih 0.9637 | validasi 0.9527
C = 28.48036  ->  latih 0.9830 | validasi 0.9645
C = 100.00000  ->  latih 0.9911 | validasi 0.9675


In [32]:
# TODO (c): temukan nilai C dengan akurasi VALIDASI tertinggi
idx_terbaik = int(np.argmax(acc_val_list))
C_terbaik = C_vals[idx_terbaik]

print("C terbaik              : {:.5f}".format(C_terbaik))
print("Akurasi latih di C itu : {:.4f}".format(acc_tr_list[idx_terbaik]))
print("Akurasi validasi       : {:.4f}".format(acc_val_list[idx_terbaik]))
print("Selisih latih-validasi : {:.4f}".format(
    acc_tr_list[idx_terbaik] - acc_val_list[idx_terbaik]))

# Latih ulang model terbaik
model_lr = Pipeline([
    ("pra", prapemrosesan),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED, C=C_terbaik)),
]).fit(X_tr, y_tr)

C terbaik              : 100.00000
Akurasi latih di C itu : 0.9911
Akurasi validasi       : 0.9675
Selisih latih-validasi : 0.0237


### 5.3 Membandingkan dengan keluarga model lain

Pada `lec03` kita membandingkan regresi logistik dengan jaringan saraf. Untuk data tabular, pembanding yang lebih lazim dan biasanya lebih kuat adalah **Random Forest** — sebuah model non-linear berbasis pohon keputusan.

In [33]:
t0 = time.time()

model_rf = Pipeline([
    ("pra", prapemrosesan),
    ("model", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
]).fit(X_tr, y_tr)

waktu_rf = time.time() - t0

acc_tr_rf = accuracy_score(y_tr, model_rf.predict(X_tr))
acc_val_rf = accuracy_score(y_val, model_rf.predict(X_val))

tabel_model = pd.DataFrame([
    {"Model": "Regresi Logistik", "Akurasi latih": acc_tr_list[idx_terbaik],
     "Akurasi validasi": acc_val_list[idx_terbaik], "Lama latih (detik)": waktu_lr},
    {"Model": "Random Forest", "Akurasi latih": acc_tr_rf,
     "Akurasi validasi": acc_val_rf, "Lama latih (detik)": waktu_rf},
])
tabel_model["Selisih latih - validasi"] = (
    tabel_model["Akurasi latih"] - tabel_model["Akurasi validasi"])

tabel_model.round(4)

,Model,Akurasi latih,Akurasi validasi,Lama latih (detik),Selisih latih - validasi
0,Regresi Logistik,0.9911,0.9675,0.2002,0.0237
1,Random Forest,1.0000,0.9438,0.7093,0.0562


### 5.4 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| Nilai `C` terbaik | 100.00000 |
| Akurasi validasi pada `C` terbaik | 0.9675 |
| Akurasi validasi pada `C` terkecil (0,0001) | 0.3018 |
| Akurasi validasi pada `C` terbesar (100) | 0.9675 |

| Model | Akurasi latih | Akurasi validasi | Lama latih (detik) | Selisih latih − validasi |
|---|---|---|---|---|
| Regresi Logistik | 0.9911 | 0.9675 | 0.2 | 0.0237 |
| Random Forest | 1.0000 | 0.9438 | 0.7093 | 0.0562 |

### 5.5 Pertanyaan Analisis

**E1.** Perhatikan grafik pada sel 5.2. Apakah kurva validasi menunjukkan pola naik-lalu-turun seperti pada teori? Bila tidak, menurut Anda mengapa?

> *Jawaban Anda:*
>
> Tidak, kurva validasi tidak menunjukkan pola naik-lalu-turun. Kurva justru terus naik seiring bertambahnya nilai C, dan mencapai titik tertinggi di ujung kanan grafik (C = 100). Hal ini terjadi karena rentang nilai C yang diuji (0.0001 sampai 100) belum cukup besar untuk mencapai titik di mana model mulai overfitting. Dengan kata lain, pada dataset ini, regularisasi yang lemah (C besar) masih memberikan performa validasi yang lebih baik, sehingga titik optimal belum tercapai dalam rentang yang diuji.

**E2.** Di sisi mana grafik itu menunjukkan **underfitting**, dan di sisi mana **overfitting**? Sebutkan kisaran nilai `C`-nya.

> *Jawaban Anda:*
>
> Underfitting terjadi pada sisi kiri grafik, yaitu pada kisaran nilai C yang sangat kecil (C < 0.01). Di sini model terlalu sederhana karena regularisasi terlalu kuat, sehingga akurasi latih dan validasi sama-sama rendah (sekitar 0.30). Overfitting belum terlihat jelas pada grafik ini karena kurva validasi masih terus naik hingga C = 100. Namun, secara teori, overfitting akan terjadi jika nilai C dinaikkan lebih jauh lagi (misalnya C > 1000) di mana akurasi latih akan terus naik mendekati 1.0 tetapi akurasi validasi mulai menurun.

**E3.** Model mana yang lebih unggul pada data validasi, dan berapa selisihnya? Apakah keunggulan itu sepadan dengan tambahan waktu pelatihannya?

> *Jawaban Anda:*
>
> Model Regresi Logistik lebih unggul pada data validasi dengan akurasi 0.9675, sedangkan Random Forest hanya 0.9438. Selisihnya sebesar 0.0237 atau sekitar 2.37 poin persen. Keunggulan ini tidak sepadan dengan tambahan waktu pelatihan, karena Regresi Logistik hanya membutuhkan 0.2 detik sedangkan Random Forest membutuhkan 0.7093 detik (sekitar 3.5 kali lebih lama). Jadi, Regresi Logistik lebih efisien dan akurat pada dataset ini.

**E4.** Bandingkan kolom "Selisih latih − validasi" kedua model. Model mana yang lebih rawan overfitting pada dataset Anda? Jelaskan mengapa hal itu masuk akal.

> *Jawaban Anda:*
>
> Random Forest lebih rawan overfitting karena memiliki selisih latih-validasi yang lebih besar (0.0562) dibandingkan Regresi Logistik (0.0237). Hal ini masuk akal karena Random Forest adalah model non-linear berbasis pohon keputusan yang mampu tumbuh sangat dalam hingga menghafal setiap detail data latih (akurasi latih mencapai 1.0000 atau sempurna), sehingga performanya di data validasi tidak sebaik di data latih. Sementara itu, Regresi Logistik memiliki regularisasi (C=100) yang membatasi kompleksitas model, sehingga selisihnya lebih kecil.


---
# Kegiatan 6 — Evaluasi Model

**Rujukan:** bagian "Evaluating the Model" pada `lec03`.

Kita menilai model terbaik pada **data uji** — yang sampai detik ini belum pernah disentuh. Ingat, data uji hanya boleh dipakai **sekali**.


### 6.1 Memilih model terbaik dan mengujinya

Lengkapi bagian bertanda `____`.

In [34]:
# TODO (a): pilih model dengan akurasi VALIDASI tertinggi
if acc_val_rf > acc_val_list[idx_terbaik]:
    model_final, nama_final = model_rf, "Random Forest"
else:
    model_final, nama_final = model_lr, "Regresi Logistik"

print("Model terpilih:", nama_final)

# TODO (b): buat prediksi pada data UJI
y_pred = model_final.predict(X_te)

acc_uji = accuracy_score(y_te, y_pred)
f1_makro = f1_score(y_te, y_pred, average="macro")

print("\nAkurasi data uji : {:.4f}".format(acc_uji))
print("F1 makro          : {:.4f}".format(f1_makro))

Model terpilih: Regresi Logistik

Akurasi data uji : 0.9409
F1 makro          : 0.9388


### 6.2 Membandingkan dengan baseline

Angka akurasi tidak berarti apa-apa tanpa pembanding. Kita pakai dua baseline:

* **Tebakan acak** — menebak kelas secara acak merata.
* **Kelas mayoritas** — selalu menjawab kelas yang paling sering muncul.

Baseline kedua sangat penting bila kelas Anda timpang.

In [35]:
dummy_acak = DummyClassifier(strategy="uniform", random_state=SEED).fit(X_tr, y_tr)
dummy_mayoritas = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

acc_acak = accuracy_score(y_te, dummy_acak.predict(X_te))
acc_mayoritas = accuracy_score(y_te, dummy_mayoritas.predict(X_te))

tabel_baseline = pd.DataFrame([
    {"Pendekatan": nama_final, "Akurasi uji": acc_uji,
     "F1 makro": f1_makro},
    {"Pendekatan": "Baseline: kelas mayoritas", "Akurasi uji": acc_mayoritas,
     "F1 makro": f1_score(y_te, dummy_mayoritas.predict(X_te), average="macro", zero_division=0)},
    {"Pendekatan": "Baseline: tebakan acak", "Akurasi uji": acc_acak,
     "F1 makro": f1_score(y_te, dummy_acak.predict(X_te), average="macro", zero_division=0)},
])

display(tabel_baseline.round(4))
print("Model unggul {:.1f} poin persen atas baseline kelas mayoritas.".format(
    (acc_uji - acc_mayoritas) * 100))

,Pendekatan,Akurasi uji,F1 makro
0,Regresi Logistik,0.9409,0.9388
1,Baseline: kelas mayoritas,0.1655,0.0406
2,Baseline: tebakan acak,0.1466,0.1437


Model unggul 77.5 poin persen atas baseline kelas mayoritas.


### 6.3 Confusion matrix

In [36]:
kelas_urut = sorted(y_te.unique())
cm = confusion_matrix(y_te, y_pred, labels=kelas_urut)

fig = px.imshow(cm, color_continuous_scale="Blues", text_auto=True,
                title="Confusion Matrix pada Data Uji — {}".format(nama_final),
                height=max(420, 60 * len(kelas_urut)))
fig.update_layout(
    xaxis_title="Label prediksi", yaxis_title="Label sebenarnya",
    coloraxis_showscale=False,
    xaxis=dict(tickmode="array", tickvals=np.arange(len(kelas_urut)), ticktext=kelas_urut),
    yaxis=dict(tickmode="array", tickvals=np.arange(len(kelas_urut)), ticktext=kelas_urut))
fig

In [37]:
# Lima kekeliruan terbesar
cm_luar = cm.copy()
np.fill_diagonal(cm_luar, 0)

n_amb = min(5, int((cm_luar > 0).sum()))
if n_amb > 0:
    urut = np.dstack(np.unravel_index(np.argsort(-cm_luar, axis=None), cm.shape))[0][:n_amb]
    daftar_keliru = pd.DataFrame([
        {"Label sebenarnya": kelas_urut[i], "Diprediksi sebagai": kelas_urut[j],
         "Jumlah": int(cm[i, j])} for i, j in urut])
    display(daftar_keliru)
else:
    print("Tidak ada kekeliruan sama sekali. Periksa kembali apakah terjadi kebocoran data!")

,Label sebenarnya,Diprediksi sebagai,Jumlah
0,Overweight_Level_II,Overweight_Level_I,5
1,Normal_Weight,Insufficient_Weight,4
2,Overweight_Level_I,Overweight_Level_II,4
3,Normal_Weight,Overweight_Level_I,4
4,Overweight_Level_I,Normal_Weight,3


### 6.4 Metrik per kelas

In [39]:
# TODO (c): buat laporan klasifikasi pada data uji
laporan = classification_report(y_te, y_pred, output_dict=True, zero_division=0)

tabel_metrik = (pd.DataFrame(laporan).transpose()
                  .loc[kelas_urut, ["precision", "recall", "f1-score", "support"]]
                  .sort_values("f1-score"))

display(tabel_metrik.round(3))

px.bar(tabel_metrik.reset_index(), x="index", y="f1-score",
       title="F1-Score per Kelas (data uji)",
       labels={"index": "Kelas", "f1-score": "F1-Score"},
       height=440).update_xaxes(categoryorder="total ascending")

,precision,recall,f1-score,support
Overweight_Level_I,0.850,0.879,0.864,58.0
Normal_Weight,0.926,0.862,0.893,58.0
Overweight_Level_II,0.930,0.914,0.922,58.0
Insufficient_Weight,0.930,0.981,0.955,54.0
Obesity_Type_II,0.967,0.967,0.967,60.0
Obesity_Type_I,0.972,0.986,0.979,70.0
Obesity_Type_III,1.000,0.985,0.992,65.0


### 6.5 Tabel Hasil Pengamatan

| Aspek | Nilai |
|---|---|
| Model terpilih | Regresi Logistik |
| Akurasi data uji | 0.9409 |
| F1 makro data uji | 0.9388 |
| Akurasi baseline kelas mayoritas | 0.1655 |
| Akurasi baseline tebakan acak | 0.1466 |
| Selisih model − baseline mayoritas | 0.7754 |

| | Kelas | Precision | Recall | F1-Score |
|---|---|---|---|---|
| **F1 terendah** | F1 terendah (Overweight_Level_I) | 0.850 | 0.879 | 0.864 |
| **F1 tertinggi** | F1 tertinggi (Obesity_Type_III) | 1.000 | 0.985 | 0.992 |

| Peringkat | Label sebenarnya | Diprediksi sebagai | Jumlah |
|---|---|---|---|
| 1 | Overweight_Level_II | Overweight_Level_I | 5 |
| 2 | Normal_Weight | Insufficient_Weight | 4 |

### 6.6 Pertanyaan Analisis

**F1.** Apakah model Anda benar-benar lebih baik daripada baseline kelas mayoritas? Tunjukkan angkanya. Bila selisihnya kecil, apa artinya?

> *Jawaban Anda:*
>
> Ya, model jauh lebih baik daripada baseline kelas mayoritas. Akurasi model mencapai 0.9409, sedangkan baseline kelas mayoritas hanya 0.1655. Selisihnya sangat besar, yaitu 0.7754 (atau 77.5 poin persen). Karena selisihnya sangat besar, ini berarti model benar-benar belajar dari pola data dan tidak sekadar menebak kelas mayoritas. Model ini sangat berguna untuk memprediksi tingkat obesitas.

**F2.** Bandingkan **akurasi** dengan **F1 makro** pada model Anda. Bila keduanya berbeda jauh, apa penyebabnya? Metrik mana yang lebih jujur untuk dataset Anda?

> *Jawaban Anda:*
>
> Akurasi (0.9409) dan F1 makro (0.9388) memiliki nilai yang sangat dekat, tidak berbeda jauh. Penyebabnya adalah distribusi kelas pada dataset ini cukup merata (meskipun secara teknis tidak seimbang sempurna). Karena tidak ada kelas yang sangat minoritas, performa model di setiap kelas relatif merata. Meskipun begitu, F1 makro tetap lebih jujur untuk dataset ini karena F1 makro menghitung rata-rata performa di semua kelas secara adil, sehingga jika ada satu kelas yang performanya buruk (misalnya F1 rendah), hal itu akan terlihat jelas dan tidak tertutupi oleh kelas lain yang performanya tinggi.

**F3.** Kelas mana yang paling sulit bagi model (F1 terendah)? Kaitkan dengan sebaran kelas pada Kegiatan 2.4 — apakah kelas itu memang jarang muncul?

> *Jawaban Anda:*
>
> Kelas yang paling sulit bagi model adalah Overweight_Level_I dengan F1-score terendah (0.864), diikuti oleh Normal_Weight (0.893). Jika dilihat dari sebaran kelas pada Kegiatan 2.4, kelas Overweight_Level_I sebenarnya bukan kelas yang jarang muncul (memiliki 58 sampel di data uji, sama seperti kelas lainnya). Kesulitan ini bukan karena jumlahnya sedikit, melainkan karena karakteristik fitur antar kelas yang berdekatan (seperti Overweight_Level_I dan Overweight_Level_II) sangat mirip, sehingga model sulit membedakannya.

**F4.** Lihat dua kekeliruan terbesar pada sel 6.3. Apakah kekeliruan itu masuk akal dari sudut pandang domainnya? (Contoh: pada data medis, dua kondisi dengan gejala mirip memang wajar tertukar.)

> *Jawaban Anda:*
>
> Ya, kekeliruan tersebut sangat masuk akal dari sudut pandang domain medis.

1. Overweight_Level_II diprediksi sebagai Overweight_Level_I (5 kali): Secara medis, batas antara Overweight Level I dan Level II sangat tipis (hanya berdasarkan perhitungan BMI yang selisihnya sedikit). Seseorang bisa dengan mudah salah dikategorikan ke level yang lebih rendah jika ada sedikit perbedaan pada fitur berat badan atau tinggi badan.
2. Normal_Weight diprediksi sebagai Insufficient_Weight (4 kali): Ini juga wajar karena batas antara berat badan normal dan berat badan kurang sangat rentan terhadap fluktuasi kecil pada fitur numerik.

**F5.** Kembali ke jawaban A4 pada Kegiatan 1 mengenai konsekuensi kesalahan. Berdasarkan confusion matrix, apakah model Anda melakukan jenis kesalahan yang paling merugikan itu? Apa yang akan Anda lakukan untuk menguranginya?

> *Jawaban Anda:*
>
> Pada A4, saya menyatakan bahwa false negative (memprediksi normal/prediksi lebih rendah dari kenyataan) adalah yang paling merugikan. Berdasarkan confusion matrix, model memang melakukan kesalahan jenis ini. Contohnya, ada 3 kasus Overweight_Level_I yang diprediksi sebagai Normal_Weight, dan 4 kasus Normal_Weight yang diprediksi sebagai Insufficient_Weight. Ini berbahaya karena orang yang sebenarnya berisiko obesitas malah diberi tahu bahwa berat badannya normal.
Untuk menguranginya, saya akan mencoba menggunakan class_weight='balanced' pada model Logistic Regression atau Random Forest, yang akan memberikan penalti lebih besar pada kesalahan klasifikasi kelas minoritas/berisiko. Selain itu, saya juga bisa mencoba melakukan resampling (seperti SMOTE) pada data latih untuk menyeimbangkan jumlah sampel antar kelas, sehingga model lebih sensitif terhadap kelas yang berisiko tinggi.


---
# Kegiatan 7 — Refleksi Transfer: Citra vs Tabular

Bagian ini menguji apakah Anda memahami **alurnya**, bukan sekadar menghafal langkah pada `lec03`.


### 7.1 Tabel Perbandingan

*Isi tabel berikut berdasarkan pengalaman Anda mengerjakan kedua praktikum.*

| Aspek | Fashion-MNIST (`lec03`) | Dataset UCI Anda |
|---|---|---|
| Jenis data | Citra grayscale 28×28 | Tabular (campuran numerik & kategorikal) |
| Jumlah baris | 60.000 (dipakai 12.000) | 2111 |
| Jumlah fitur setelah featurisasi | 784 | 16 fitur awal (30 setelah one-hot encoding) |
| Tipe fitur | Seragam (intensitas piksel) | Campuran (numerik kontinu & kategorikal) |
| Nilai hilang | Tidak ada | Tidak ada |
| Keseimbangan kelas | Seimbang sempurna | Tidak seimbang (rasio ~1.3 : 1) |
| Langkah pra-pemrosesan | Flatten + standarisasi | Imputasi + Standarisasi + One-Hot Encoding |
| Model terbaik | Jaringan Saraf (Neural Network) | Regresi Logistik |
| Akurasi uji | (Sesuai lec03) | 0.9409 |
| Metrik yang paling sesuai | Akurasi | F1-Macro (karena kelas tidak seimbang) |


### 7.2 Pertanyaan Analisis

**G1.** Sebutkan **tiga langkah pra-pemrosesan** yang diperlukan pada dataset UCI Anda tetapi **tidak diperlukan** pada Fashion-MNIST. Jelaskan mengapa masing-masing menjadi perlu.

> *Jawaban Anda:*
>
> 1. Imputasi Nilai Hilang (SimpleImputer): Diperlukan karena data tabular dari survei seringkali memiliki nilai kosong. Pada Fashion-MNIST, semua piksel gambar selalu terisi (tidak ada nilai kosong).
2. One-Hot Encoding: Diperlukan untuk mengubah kolom kategorikal (seperti Gender, MTRANS) menjadi format numerik yang bisa diproses model. Pada Fashion-MNIST, semua fitur sudah berupa angka (intensitas piksel).
3. Standarisasi Fitur Numerik (StandardScaler): Diperlukan karena rentang nilai antar kolom numerik berbeda jauh (misal Weight ratusan vs Height satuan). Pada Fashion-MNIST, semua fitur memiliki skala yang seragam (0-255).

**G2.** Sebutkan **satu langkah** yang diperlukan pada Fashion-MNIST tetapi tidak relevan pada dataset Anda, dan jelaskan alasannya.

> *Jawaban Anda:*
>
> Flatten (perataan gambar). Langkah ini diperlukan pada Fashion-MNIST untuk mengubah matriks piksel 2D (28x28) menjadi vektor 1D (784) agar bisa dimasukkan ke model. Pada dataset tabular UCI, data sudah dalam bentuk baris dan kolom (tabel), sehingga tidak ada struktur spasial 2D yang perlu diratakan.

**G3.** Pada `lec03` jaringan saraf mengungguli regresi logistik. Pada dataset tabular Anda, apakah model non-linear (Random Forest) juga menang? Menurut Anda, mengapa untuk data tabular model berbasis pohon sering lebih unggul daripada jaringan saraf?

> *Jawaban Anda:*
>
> Tidak, Random Forest tidak menang. Regresi Logistik justru lebih unggul dengan akurasi validasi 0.9675, sedangkan Random Forest hanya 0.9438. Model berbasis pohon (seperti Random Forest) sering lebih unggul pada data tabular karena mampu menangkap hubungan non-linear dan interaksi antar fitur dengan baik, serta lebih tahan terhadap outlier dan fitur dengan skala berbeda tanpa memerlukan normalisasi yang rumit. Jaringan saraf biasanya lebih unggul pada data tidak terstruktur seperti citra atau teks, di mana pola spasial dan sekuensial sangat penting.

**G4.** Bagian mana dari siklus hidup ML (L — M — O — P) yang menurut Anda **paling banyak berubah** ketika berpindah dari data citra ke data tabular? Bagian mana yang justru **tetap sama persis**?

> *Jawaban Anda:*
>
> Bagian yang paling banyak berubah adalah P (Preprocessing/Persiapan Data) dan M (Modeling). Pada data citra, pra-pemrosesan hanya flatten dan standarisasi, sedangkan pada data tabular perlu imputasi, standarisasi, dan one-hot encoding. Model yang cocok juga berbeda (CNN vs Regresi Logistik/Random Forest). Bagian yang tetap sama adalah L (Learning Problem) dan P (Performance/Evaluation), yaitu tetap merumuskan masalah klasifikasi dan mengevaluasi dengan akurasi/F1.

**G5.** Bila diberi dataset baru lagi minggu depan, tuliskan **daftar periksa lima langkah pertama** yang akan Anda kerjakan, berdasarkan pengalaman dua praktikum ini.

> *Jawaban Anda:*
>
> 1. Memahami konteks dataset dan merumuskan masalah (target, kelas, tujuan).
> 2. Melakukan eksplorasi data (EDA): cek tipe data, nilai hilang, sebaran kelas, statistik deskriptif.
> 3. Memisahkan data latih, validasi, dan uji (dengan stratify jika perlu).
> 4. Membangun pipeline pra-pemrosesan (imputasi, scaling, encoding) yang aman dari kebocoran data.
> 5. Melatih model baseline (seperti Regresi Logistik), lalu membandingkan dengan model lain (seperti Random Forest) dan mengevaluasi metrik yang sesuai.


---
# Kesimpulan dan Refleksi

### Kesimpulan

*Tuliskan minimal lima poin kesimpulan berdasarkan hasil percobaan Anda sendiri. Sebutkan angka bila relevan, dan hindari kalimat teori umum yang bisa ditulis tanpa menjalankan kode.*

1. Pra-pemrosesan data tabular yang mencakup imputasi, standarisasi, dan one-hot encoding terbukti berhasil mengubah 16 fitur awal menjadi 30 fitur siap latih (pertambahan 14 kolom), dan berhasil mencegah kebocoran data karena semua proses fit hanya dilakukan pada data latih (X_tr).
2. Model Regresi Logistik berhasil mengalahkan baseline mayoritas dengan sangat signifikan, di mana akurasi data uji mencapai 0.9409 dibandingkan baseline kelas mayoritas yang hanya 0.1655 (selisih 77.5 poin persen).
3. Penyetelan hyperparameter C pada Regresi Logistik menunjukkan bahwa nilai C terbaik adalah 100.00000 dengan akurasi validasi 0.9675, yang berarti model ini belum mencapai titik overfitting pada rentang nilai C yang diuji (0.0001 - 100).
4. Random Forest tidak selalu lebih unggul pada data tabular, di mana model ini kalah dari Regresi Logistik pada data validasi (0.9438 vs 0.9675) dan justru lebih rawan overfitting dengan selisih latih-validasi sebesar 0.0562 (Regresi Logistik hanya 0.0237).
5. Kelas Overweight_Level_I menjadi kelas tersulit bagi model dengan F1-score terendah (0.864), yang disebabkan karena karakteristiknya sangat mirip dengan kelas Overweight_Level_II (kesalahan prediksi terbanyak: 5 sampel Overweight_Level_II diprediksi sebagai Overweight_Level_I).


### Refleksi

**R1.** Kesulitan terbesar apa yang Anda hadapi saat memindahkan alur `lec03` ke dataset baru ini?

> *Jawaban Anda:*
>
> Kesulitan terbesar adalah beradaptasi dari data citra yang homogen (semua fitur berupa intensitas piksel) ke data tabular yang heterogen. Saya harus memikirkan cara menangani kolom kategorikal yang harus diubah menjadi numerik (one-hot encoding) dan memastikan kolom numerik dengan rentang nilai yang berbeda jauh (seperti Weight vs Height) distandarisasi. Selain itu, membangun pipeline yang tepat agar tidak terjadi kebocoran data (data leakage) pada data tabular membutuhkan ketelitian lebih dibandingkan saat meratakan gambar.


**R2.** Error apa yang Anda temui, dan bagaimana Anda mengatasinya? Sebutkan minimal satu.

> *Jawaban Anda:*
>
> Error yang saya temui adalah ketika saya mencoba melatih model dengan data yang sudah ditransformasi (X_tr_siap) tetapi menggunakan pipeline yang di dalamnya sudah ada prapemrosesan. Ini menyebabkan error karena pipeline mencoba melakukan fit ulang pada data yang sudah di-one-hot encoding. Saya mengatasinya dengan memasukkan data mentah (X_tr) langsung ke dalam model_lr.fit(), sehingga pipeline secara otomatis melakukan pra-pemrosesan dari awal tanpa error. Error lain adalah potensi error saat ada kategori baru di data uji, yang diatasi dengan handle_unknown='ignore' pada OneHotEncoder.


**R3.** Bila diberi waktu satu minggu lagi, apa **satu** hal pertama yang akan Anda coba untuk memperbaiki model ini? Mengapa itu yang Anda dahulukan?

> *Jawaban Anda:*
>
> Hal pertama yang akan saya coba adalah menangani ketidakseimbangan kelas dan tumpang tindih antar kelas (terutama Overweight_Level_I dan Overweight_Level_II) dengan menggunakan parameter class_weight='balanced' atau melakukan feature engineering seperti menghitung BMI (Body Mass Index) dari fitur Weight dan Height. Ini saya dahulukan karena hasil confusion matrix menunjukkan bahwa kesalahan terbesar model adalah membedakan kelas-kelas yang berdekatan (misklasifikasi level overweight), sehingga memperbaiki pemisahan antar kelas tersebut akan memberikan peningkatan akurasi yang paling signifikan dibandingkan hanya mengubah hyperparameter model.


### Daftar Rujukan

*Tuliskan sumber yang Anda pakai: halaman UCI dataset, dokumentasi scikit-learn, artikel, atau lainnya.*

1. UCI Machine Learning Repository. (2019). Estimation of Obesity Levels Based On Eating Habits and Physical Condition. Diakses dari https://archive.ics.uci.edu/dataset/544/estimation-of-obesity-levels-based-on-eating-habits-and-physical-condition
2. Scikit-learn Developers. (2024). Preprocessing data: StandardScaler, OneHotEncoder, SimpleImputer. Diakses dari https://scikit-learn.org/stable/modules/preprocessing.html
3. Scikit-learn Developers. (2024). Pipeline and ColumnTransformer. Diakses dari https://scikit-learn.org/stable/modules/compose.html



---
# Rubrik Penilaian

*Bagian ini diisi oleh dosen atau asisten praktikum.*

| No | Aspek yang Dinilai | Bobot | Skor (0–100) | Nilai |
|---|---|---|---|---|
| 1 | Ketepatan pemilihan dataset, pemenuhan kriteria K1–K9, dan alasan pemilihan | 8% | | |
| 2 | Perumusan permasalahan pembelajaran (Kegiatan 1) | 15% | | |
| 3 | Kedalaman eksplorasi data (Kegiatan 2–3) | 15% | | |
| 4 | Ketepatan pipeline pra-pemrosesan (Kegiatan 4) | 15% | | |
| 5 | Pemodelan, penyetelan, dan perbandingan model (Kegiatan 5) | 15% | | |
| 6 | Evaluasi dan pemilihan metrik (Kegiatan 6) | 15% | | |
| 7 | Refleksi transfer citra vs tabular (Kegiatan 7) | 12% | | |
| 8 | Kesimpulan, refleksi, dan rujukan | 5% | | |
| | **Nilai Akhir** | **100%** | | |

**Catatan dosen:**

> ......

### Pedoman skor jawaban analisis

| Skor | Kriteria |
|---|---|
| 85–100 | Jawaban tepat, didukung angka dari percobaan sendiri, dikaitkan dengan domain datasetnya, dan menunjukkan penalaran mandiri |
| 70–84 | Jawaban tepat tetapi kurang didukung data, atau tidak dikaitkan dengan domain dataset |
| 55–69 | Jawaban sebagian benar, atau hanya mengulang teori tanpa menyentuh hasil percobaan |
| < 55 | Jawaban tidak tepat, kosong, atau menunjukkan kemiripan mencolok dengan pekerjaan mahasiswa lain |


---
# Daftar Periksa Sebelum Mengumpulkan

Centang dengan mengganti `[ ]` menjadi `[x]` (klik dua kali sel ini untuk mengedit).

- [ ] Identitas dan **kartu klaim dataset** sudah diisi lengkap
- [ ] Dataset saya sudah didaftarkan dan dikonfirmasi berbeda dari mahasiswa lain
- [ ] `DATASET_ID` dan `NIM_3_DIGIT` sudah diisi dengan data saya sendiri
- [ ] Sel P.4 menunjukkan **semua kriteria LULUS**
- [ ] Tabel verifikasi kriteria (K1–K9) sudah diisi lengkap
- [ ] Semua sel bertanda `TODO` sudah dilengkapi dan berjalan tanpa error
- [ ] Seluruh sel sudah dijalankan berurutan dari atas ke bawah
- [ ] Semua tabel hasil pengamatan sudah diisi angka dari komputer saya
- [ ] Semua pertanyaan analisis A1 sampai G5 sudah dijawab
- [ ] Tabel perbandingan citra vs tabular (7.1) sudah lengkap
- [ ] Kesimpulan minimal lima poin dan refleksi R1–R3 sudah ditulis
- [ ] Daftar rujukan sudah diisi
- [ ] Notebook sudah disimpan (`Ctrl + S`) sebelum diekspor

### Langkah ekspor PDF

1. Simpan notebook: `Ctrl + S`
2. `File` → `Save and Export Notebook As...` → `HTML`
3. Buka berkas HTML di browser, tekan `Ctrl + P`, pilih **Save as PDF**
4. Pada dialog cetak, aktifkan **Background graphics** agar grafik ikut tercetak berwarna
5. Beri nama berkas: `LKM_UCI_NIM_NamaLengkap.pdf`

> **Bila grafik Plotly tidak muncul pada hasil ekspor,** jalankan sel di bawah lalu jalankan ulang sel-sel grafik sebelum mengekspor.


In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"
print("Renderer Plotly diatur ke 'notebook'. Jalankan ulang sel grafik, lalu ekspor kembali.")

---

*Lembar Kerja Mahasiswa — Tugas Penerapan Siklus Hidup Machine Learning*
*Kelanjutan dari praktikum `lec03` (Fashion-MNIST). Dataset bersumber dari UCI Machine Learning Repository (https://archive.ics.uci.edu/datasets).*
